In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

DATA_IMAGE = PROJECT_ROOT / "data" / "images"
DATA_IMAGE.mkdir(parents=True, exist_ok=True)

DATA_SYMBOLIC = PROJECT_ROOT / "data" / "symbolic"
DATA_SYMBOLIC.mkdir(parents=True, exist_ok=True)

print("Processed data folder:", DATA_PROCESSED)

Processed data folder: c:\Users\ccana\Documents\Doutorado\VISEMTracking\data\processed


In [3]:
df = pd.read_csv(DATA_PROCESSED / "01_processed.csv")
df = df.sort_values(["user_id", "trajectory_id"])
df

,user_id,timestamp,trajectory_id,ftid,x,y,w,h
0,11,0,ckz3v9nzv00033867jsekqdcl,0,0.275781,0.412500,0.026562,0.037500
1,11,1,ckz3v9nzv00033867jsekqdcl,0,0.275767,0.412549,0.026562,0.037500
2,11,2,ckz3v9nzv00033867jsekqdcl,0,0.275754,0.412598,0.026562,0.037500
3,11,3,ckz3v9nzv00033867jsekqdcl,0,0.275740,0.412647,0.026562,0.037500
4,11,4,ckz3v9nzv00033867jsekqdcl,0,0.275726,0.412697,0.026562,0.037500
...,...,...,...,...,...,...,...,...
586871,82,777,cl5mohjd6000e3b6g97eyecxk,0,0.576598,0.672114,0.031250,0.039583
586872,82,778,cl5mohjd6000e3b6g97eyecxk,0,0.576580,0.671995,0.031250,0.039583
586873,82,779,cl5mohjd6000e3b6g97eyecxk,0,0.576562,0.671875,0.031250,0.039583
586874,82,780,cl5mohjd6000e3b6g97eyecxk,0,0.576562,0.671875,0.031250,0.039583


In [4]:
df = df.sort_values(["trajectory_id", "timestamp"]).reset_index(drop=True)

df['x_translate'] = df.groupby('trajectory_id')['x'].transform(lambda x: x - x.iloc[0])
df['y_translate'] = df.groupby('trajectory_id')['y'].transform(lambda y: y - y.iloc[0])

angles = {}

def pca_angle(x, y):
    X = np.vstack([x, y]).T
    Xc = X - X.mean(axis=0)
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    v1 = Vt[0]
    disp = np.array([x[-1], y[-1]])
    if np.dot(v1, disp) < 0:
        v1 = -v1
    return np.arctan2(v1[1], v1[0])

df['x_rotated'] = np.nan
df['y_rotated'] = np.nan

for tid, g in df.groupby('trajectory_id'):
    x, y = g['x_translate'].to_numpy(), g['y_translate'].to_numpy()
    theta = pca_angle(x, y)
    c, s = np.cos(-theta), np.sin(-theta)
    xr = x * c - y * s
    yr = x * s + y * c
    df.loc[g.index, 'x_rotated'] = xr
    df.loc[g.index, 'y_rotated'] = yr
    angles[tid] = theta


In [5]:
def combine_labels(speed_lbl, dir_lbl):
    if pd.isna(speed_lbl):
        return ""
    if speed_lbl == "Parado":
        return "Parado"
    if pd.isna(dir_lbl):
        return ""
    return speed_lbl + "_" + dir_lbl


# -----------------------------
# Funções de rotulagem
# -----------------------------
def label_dir(a):
    if np.isnan(a):
        return np.nan

    a = ((a + 180) % 360) - 180

    if -45 < a <= 45:
        return "Leste"
    elif 45 < a <= 135:
        return "Norte"
    elif a > 135 or a <= -135:
        return "Oeste"
    else:
        return "Sul"


def label_speed_likert(v, q20, q40, q60, q80):
    if np.isnan(v):
        return np.nan
    if v <= q20:
        return "Muito_Lento"
    elif v <= q40:
        return "Lento"
    elif v <= q60:
        return "Medio"
    elif v <= q80:
        return "Rapido"
    else:
        return "Muito_Rapido"


In [ ]:


# -----------------------------
# DataFrame simbólico base
# -----------------------------
df_symbol = pd.DataFrame({"trajectory_id": df["trajectory_id"].unique()})


# -----------------------------
# Suavizações
# -----------------------------
k = [10, 15, 20]

for n in k:
    df_suavizado = (
        df[df["timestamp"] % n == 0]
        .sort_values(["trajectory_id", "timestamp"])
        .reset_index(drop=True)
    )

    print(
        f"Suavização de {n} frames.\n"
        f"Antes: {df.shape}\n"
        f"Depois: {df_suavizado.shape}\n"
    )

    # -----------------------------
    # Diferenças espaciais
    # -----------------------------
    dx = df_suavizado.groupby("trajectory_id")["x_rotated"].diff()
    dy = df_suavizado.groupby("trajectory_id")["y_rotated"].diff()

    df_suavizado[f"speed_{n}"] = np.hypot(dx, dy)
    df_suavizado[f"angle_deg_{n}"] = np.degrees(np.arctan2(dy, dx))

    # -----------------------------
    # Quantis
    # -----------------------------
    valid_speed = df_suavizado[f"speed_{n}"].dropna()

    if not valid_speed.empty:
        q20, q40, q60, q80 = np.quantile(
            valid_speed, [0.20, 0.40, 0.60, 0.80]
        )
    else:
        q20 = q40 = q60 = q80 = 0.0

    # -----------------------------
    # Rotulagem
    # -----------------------------
    df_suavizado[f"speed_label_{n}"] = df_suavizado[f"speed_{n}"].apply(
        lambda v: label_speed_likert(v, q20, q40, q60, q80)
    )

    df_suavizado[f"dir4_label_{n}"] = df_suavizado[f"angle_deg_{n}"].apply(label_dir)

    df_suavizado[f"symbol_{n}"] = [
        combine_labels(s, d)
        for s, d in zip(
            df_suavizado[f"speed_label_{n}"],
            df_suavizado[f"dir4_label_{n}"]
        )
    ]

    # -----------------------------
    # Agregação simbólica 
    # -----------------------------
    df_suavizado = df_suavizado[
        df_suavizado[f"symbol_{n}"].notna() &
        (df_suavizado[f"symbol_{n}"] != "")
    ]
    symbolic_n = (
        df_suavizado
        .groupby("trajectory_id")[f"symbol_{n}"]
        .agg(list)
        .reset_index(name=f"symbolic_movement_{n}")
    )

    # Merge no dataframe simbólico
    df_symbol = df_symbol.merge(
        symbolic_n,
        on="trajectory_id",
        how="left"
    )

df_symbol.to_csv(DATA_SYMBOLIC / "02_symbolic_per_trajectory.csv", index=False)
df_symbol


Suavização de 10 frames.
Antes: (586876, 12)
Depois: (58722, 12)

Suavização de 15 frames.
Antes: (586876, 12)
Depois: (39151, 12)

Suavização de 20 frames.
Antes: (586876, 12)
Depois: (29521, 12)



,trajectory_id,symbolic_movement_10,symbolic_movement_15,symbolic_movement_20
0,ckyw6zzlj001r3867thf0fuy7,"[Muito_Rapido_Leste, Muito_Rapido_Leste, Muito...","[Muito_Rapido_Leste, Muito_Rapido_Norte, Muito...","[Muito_Rapido_Leste, Muito_Rapido_Norte, Muito..."
1,ckyw704kw001v3867kvyjtx6k,"[Muito_Rapido_Oeste, Muito_Rapido_Oeste, Muito...","[Muito_Rapido_Oeste, Muito_Rapido_Sul, Muito_R...","[Muito_Rapido_Oeste, Muito_Rapido_Sul, Muito_R..."
2,ckyw708pn001z386779fr849h,"[Rapido_Leste, Rapido_Leste, Rapido_Leste, Rap...","[Rapido_Leste, Rapido_Leste, Rapido_Leste, Rap...","[Rapido_Leste, Rapido_Leste, Rapido_Oeste, Med..."
3,ckyw7dzmx00273867fyj9lsnv,"[Muito_Rapido_Norte, Muito_Rapido_Norte, Muito...","[Muito_Rapido_Norte, Muito_Rapido_Norte, Muito...","[Muito_Rapido_Norte, Muito_Rapido_Oeste, Muito..."
4,ckyw7e4o6002b3867ibaukox2,"[Rapido_Norte, Rapido_Leste, Rapido_Leste, Rap...","[Rapido_Norte, Rapido_Leste, Rapido_Leste, Rap...","[Rapido_Leste, Rapido_Leste, Rapido_Leste, Rap..."
...,...,...,...,...
802,cl691zdqu000h3b6gpesz4820,"[Medio_Leste, Medio_Leste, Medio_Leste, Medio_...","[Medio_Leste, Medio_Leste, Medio_Leste, Medio_...","[Medio_Leste, Medio_Leste, Medio_Leste, Medio_..."
803,cl696haqz00073b6gf0un301j,"[Muito_Lento_Leste, Muito_Lento_Leste, Muito_L...","[Muito_Lento_Leste, Muito_Lento_Leste, Muito_L...","[Muito_Lento_Leste, Muito_Lento_Leste, Muito_L..."
804,cl696kim5000j3b6g3waaz3ij,"[Muito_Lento_Leste, Muito_Lento_Leste, Muito_L...","[Muito_Lento_Leste, Muito_Lento_Leste, Muito_L...","[Muito_Lento_Leste, Muito_Lento_Leste, Muito_L..."
805,cl696ldsk000n3b6g3cevz2tg,"[Muito_Lento_Leste, Muito_Lento_Leste, Muito_L...","[Muito_Lento_Leste, Muito_Lento_Leste, Muito_L...","[Muito_Lento_Leste, Muito_Lento_Leste, Muito_L..."
